## Library Imports and Configs 

In [66]:
import numpy as np 
import pandas as pd 
pd.set_option('display.max_columns', None)
import os
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns

ROOT_PATH = "playground-series-s6e5"

train_df = pd.read_csv(os.path.join(ROOT_PATH, "train.csv"))
test_df = pd.read_csv(os.path.join(ROOT_PATH, "test.csv"))
sub_df = pd.read_csv(os.path.join(ROOT_PATH, "sample_submission.csv"))

train_df.shape, test_df.shape # (70 % train, 30 % test)

((439140, 16), (188165, 15))

In [67]:
train_df.sample(4)

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
396272,396272,D029,HARD,Emilia Romagna Grand Prix,2024,0,30,2,17.0,13,82.216,-0.646,-36.494,0.422535,3.0,0.0
6941,6941,D035,HARD,Saudi Arabian Grand Prix,2022,1,34,2,28.0,4,94.835,-8.908,-86.741,0.472222,2.0,1.0
118793,118793,D009,HARD,Spanish Grand Prix,2023,0,65,2,38.0,4,78.839,-0.084,-20.848,0.984848,0.0,0.0
404530,404530,CHI,HARD,Miami Grand Prix,2022,0,24,2,12.0,7,94.924,-31.599,-53.345,0.333333,3.0,0.0


In [68]:
race_places = set()
for race in train_df['Race']:
    if 'united states' in race.lower():
        race_places.add(race)


race_places

{'United States Grand Prix'}

In [69]:
target_counts = train_df['PitNextLap'].value_counts().to_dict()
print("Percentage count of 0.0 PitNextLap")
print(target_counts[0.0] / sum(list(target_counts.values())) * 100)
print("Percentage count of 1.0 PitNextLap")
print(target_counts[1.0] / sum(list(target_counts.values())) * 100)

Percentage count of 0.0 PitNextLap
80.101789862003
Percentage count of 1.0 PitNextLap
19.898210137996994


## Feature Engineering
- Domain Knowledge
- Correlation after feature engineering

In [70]:
def engineer_race_features(df):
    """
    Applies feature engineering for race strategy prediction.
    Handles high correlation via ratios and extracts stint-based metrics.
    """
    # Create a copy to avoid SettingWithCopyWarning
    df = df.copy()

    # 1. Handling your high correlation pair (RaceProgress / LapNumber)
    # Adding a small epsilon to avoid division by zero
    df['Progress_Per_Lap_engg'] = df['RaceProgress'] / (df['LapNumber'] + 1e-5)

    # 2. Tyre & Degradation Ratios
    # Captures the intensity of degradation relative to the distance traveled
    df['Deg_Per_Lap_engg'] = df['Cumulative_Degradation'] / (df['LapNumber'] + 1e-5)
    df['Deg_Per_TyreLife_engg'] = df['Cumulative_Degradation'] / (df['TyreLife'] + 1e-5)

    # 4. Pace Sensitivity (The "Cliff" Detector)
    # How much is the lap time changing relative to tyre age?
    df['Pace_Tyre_Sensitivity_engg'] = df['LapTime_Delta'] / (df['TyreLife'] + 1e-5)

    # 5. Strategic Flags
    # Identify if a driver is losing positions (potential pressure to pit)
    df['Losing_Ground_engg'] = (df['Position_Change'] < 0).astype(int)
    
    # Identify 'Fresh' vs 'Old' tyres based on stint start
    df['Is_Late_Stint_engg'] = (df['TyreLife'] > 20).astype(int) 

    # 6. Interaction Terms
    # Multiplying LapTime_Delta by Cumulative_Degradation to highlight 
    # laps where both pace drops and wear is high
    df['Wear_Pace_Impact_engg'] = df['LapTime_Delta'] * df['Cumulative_Degradation']

    # # position change per lap number
    # df['Pos_Change_Per_Lap_engg'] = df['Position_Change'] / df['LapTime (s)']

    return df


# Apply to your dataframes
train_df_eng = engineer_race_features(train_df)
test_df_eng = engineer_race_features(test_df)

train_df_eng.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap,Progress_Per_Lap_engg,Deg_Per_Lap_engg,Deg_Per_TyreLife_engg,Pace_Tyre_Sensitivity_engg,Losing_Ground_engg,Is_Late_Stint_engg,Wear_Pace_Impact_engg
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0,0.014286,0.420380,0.538949,-0.193949,0,1,-158.987716
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0,0.012821,-8.266923,-31.886669,-4.659565,1,0,7280.342719
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0,0.013889,-1.703881,-4.569498,-0.342727,0,1,757.988660
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0,0.038461,-3.661982,-3.661982,-3.661982,0,0,53.640976
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0,0.013889,-0.543807,-2.356496,1.494164,0,0,-126.756135


In [71]:
cat_cols = test_df_eng.drop(['id'], axis='columns').select_dtypes(exclude=np.number).columns
num_cols = test_df_eng.drop(['id'], axis='columns').select_dtypes(include=np.number).columns
target_col = ['PitNextLap']

## Data Processing
- Encoding - (categorical data) (Label Encoding and OneHotEncoding)
- Normalization - (numerical data)

#### Encoding - Categorical data - Label Encoding and OneHotEncoding

In [72]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

label_encoder = LabelEncoder()

train_df_eng["Driver_en"] = label_encoder.fit_transform(train_df_eng['Driver'])
test_df_eng["Driver_en"] = label_encoder.transform(test_df_eng['Driver'])

train_df_eng["Race_en"] = label_encoder.fit_transform(train_df_eng['Race'])
test_df_eng["Race_en"] = label_encoder.transform(test_df_eng['Race'])

    
train_df_eng_enc = train_df_eng.drop(["Driver", "Race"], axis='columns')
test_df_eng_enc = test_df_eng.drop(["Driver", "Race"], axis='columns')

In [73]:
one_hot_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoded_data = one_hot_encoder.fit_transform(train_df_eng_enc[['Compound']])
encoded_df = pd.DataFrame(
    encoded_data, 
    columns=one_hot_encoder.get_feature_names_out(['Compound']),
    index=train_df_eng_enc.index
)
train_df_eng_enc_v1 = pd.concat([train_df_eng_enc, encoded_df], axis=1).drop('Compound', axis=1)


test_encoded_data = one_hot_encoder.transform(test_df_eng_enc[['Compound']])
test_encoded_df = pd.DataFrame(
    test_encoded_data, 
    columns=one_hot_encoder.get_feature_names_out(['Compound']),
    index=test_df_eng_enc.index
)
test_df_eng_enc_v1 = pd.concat([test_df_eng_enc, test_encoded_df], axis=1).drop('Compound', axis=1)

train_df_eng_enc_v1.filter(like='Compound').head()

,Compound_HARD,Compound_INTERMEDIATE,Compound_MEDIUM,Compound_SOFT,Compound_WET
0,1.0,0.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0
2,1.0,0.0,0.0,0.0,0.0
3,0.0,0.0,1.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0


#### Normalization (MinMax / StandardScaler)

In [74]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

class DataNormalizer:
    def __init__(self, method='standard'):
        """
        Initializes the normalizer with the chosen scaling strategy.
        method: 'minmax' or 'standard'
        """
        self.method = method
        if method == 'minmax':
            self.scaler = MinMaxScaler()
        elif method == 'standard':
            self.scaler = StandardScaler()
        else:
            raise ValueError("Method must be either 'minmax' or 'standard'")

    def fit(self, dataset, num_cols):
        """
        Learns the scaling parameters (mean/std or min/max) from the training set.
        """
        if not all(col in dataset.columns for col in num_cols):
            missing = [c for c in num_cols if c not in dataset.columns]
            raise ValueError(f"Columns missing from dataset: {missing}")
            
        self.scaler.fit(dataset[num_cols])
        print(f"Successfully fitted {self.method} scaler on: {num_cols}")

    def transform(self, dataset, num_cols):
        """
        Applies the learned parameters to scale the dataset.
        """
        df = dataset.copy()
        df[num_cols] = self.scaler.transform(df[num_cols])
        return df

    def fit_transform(self, dataset, num_cols):
        """
        Fits to the data then transforms it. Useful for the initial training set.
        """
        self.fit(dataset, num_cols)
        return self.transform(dataset, num_cols)


normalizer = DataNormalizer(method='standard')

train_df_scaled = normalizer.fit_transform(train_df_eng_enc_v1, num_cols=num_cols)
test_df_scaled = normalizer.transform(test_df_eng_enc_v1, num_cols=num_cols)

train_df_scaled.head()

Successfully fitted standard scaler on: Index(['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position',
       'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation',
       'RaceProgress', 'Position_Change', 'Progress_Per_Lap_engg',
       'Deg_Per_Lap_engg', 'Deg_Per_TyreLife_engg',
       'Pace_Tyre_Sensitivity_engg', 'Losing_Ground_engg',
       'Is_Late_Stint_engg', 'Wear_Pace_Impact_engg'],
      dtype='str')


,id,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap,Progress_Per_Lap_engg,Deg_Per_Lap_engg,Deg_Per_TyreLife_engg,Pace_Tyre_Sensitivity_engg,Losing_Ground_engg,Is_Late_Stint_engg,Wear_Pace_Impact_engg,Driver_en,Race_en,Compound_HARD,Compound_INTERMEDIATE,Compound_MEDIUM,Compound_SOFT,Compound_WET
0,0,-1.486487,-0.396946,1.585901,0.221941,2.534531,-0.308849,-0.630046,-0.086333,0.853455,1.487008,1.222548,1.0,-0.091033,0.151346,0.220112,0.024664,-0.688951,1.816110,-0.019963,134,7,1.0,0.0,0.0,0.0,0.0
1,1,1.440545,2.519236,0.229628,0.221941,-0.730333,-1.066602,-0.801797,-0.656423,-3.605949,0.033534,-0.774077,0.0,-0.401788,-0.384754,-1.788677,-0.088712,1.451482,-0.550627,0.365547,111,9,1.0,0.0,0.0,0.0,0.0
2,2,-1.486487,-0.396946,2.116616,1.274359,0.800072,0.638343,-1.011682,-0.085787,-1.365930,1.902200,0.723392,1.0,-0.175196,0.020257,-0.096360,0.020887,-0.688951,1.816110,0.027555,886,2,1.0,0.0,0.0,0.0,0.0
3,3,-0.510810,-0.396946,-1.244581,-0.830476,-1.240468,-0.498287,0.172574,-0.080872,0.335931,-1.029455,-0.025343,0.0,5.036382,-0.100579,-0.040139,-0.063385,-0.688951,-0.550627,-0.008944,864,19,0.0,0.0,1.0,0.0,0.0
4,4,-1.486487,2.519236,0.170660,1.274359,-0.832360,-1.445478,0.856192,0.289790,0.211493,0.092589,0.723392,0.0,-0.175196,0.091846,0.040737,0.067523,-0.688951,-0.550627,-0.018293,44,3,1.0,0.0,0.0,0.0,0.0


In [75]:
test_df_scaled.head()

,id,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,Progress_Per_Lap_engg,Deg_Per_Lap_engg,Deg_Per_TyreLife_engg,Pace_Tyre_Sensitivity_engg,Losing_Ground_engg,Is_Late_Stint_engg,Wear_Pace_Impact_engg,Driver_en,Race_en,Compound_HARD,Compound_INTERMEDIATE,Compound_MEDIUM,Compound_SOFT,Compound_WET
0,439140,-0.510810,-0.396946,-0.124182,-0.830476,0.698045,-1.066602,0.123314,0.092160,0.378658,0.261317,-0.025343,0.957763,0.110758,0.172020,0.029926,-0.688951,1.816110,-0.011796,144,6,0.0,0.0,1.0,0.0,0.0
1,439141,-0.510810,-0.396946,0.052723,-0.830476,1.004126,-1.634917,-0.004134,0.082853,0.433326,0.300590,-0.025343,0.535834,0.120288,0.181587,0.029451,-0.688951,1.816110,-0.011711,875,0,0.0,0.0,1.0,0.0,0.0
2,439142,-0.510810,-0.396946,0.052723,-0.830476,1.004126,0.259466,0.097218,0.086722,0.308213,0.489100,-0.025343,0.957763,0.102669,0.163900,0.029631,-0.688951,1.816110,-0.011743,295,6,0.0,0.0,1.0,0.0,0.0
3,439143,0.464868,-0.396946,-1.008708,0.221941,-1.036414,1.017219,0.203222,-0.363425,0.620302,-1.025511,0.224236,-0.366479,0.210257,0.314496,-0.095711,-0.688951,-0.550627,-0.020164,137,24,0.0,0.0,0.0,1.0,0.0
4,439144,0.464868,-0.396946,1.703838,0.221941,1.514261,0.448905,0.412854,0.106951,0.088992,1.518343,1.721705,-0.175196,0.100663,0.142187,0.030402,-0.688951,1.816110,-0.012729,4,25,1.0,0.0,0.0,0.0,0.0


In [76]:
train_df_scaled.shape, test_df_scaled.shape

((439140, 27), (188165, 26))

## Choosing Best Model 

In [77]:
def submission(model, test_data, file_name:str):
    pred_data = test_data
    if 'id' in test_data.columns:
        pred_data = test_data.drop(['id'], axis='columns')
        print("!!!!!!!!!!!!!!!!!! Removed id column !!!!!!!!!!!!!!!!!!!")
        
    predictions = model.predict_proba(pred_data)[:, 1]
    sub_df['PitNextLap'] = predictions
    sub_df[['id', 'PitNextLap']].to_csv(file_name, index=False)
    print(f"Submissions saved to {file_name} path!!!!!!!!!!!!!!")

In [91]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [92]:
X, y = train_df_scaled.drop(['id', 'PitNextLap'], axis='columns'), train_df_scaled['PitNextLap'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

lgb_model = LGBMClassifier(
    n_estimators      = 2000,
    learning_rate     = 0.02,
    num_leaves        = 255,
    min_child_samples = 20,
    subsample         = 0.8,
    subsample_freq    = 1,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 0.1,
    random_state      = 42,
    n_jobs            = -1
)
xgb_model = XGBClassifier(
    n_estimators      = 2000,
    learning_rate     = 0.02,
    max_depth         = 7,
    min_child_weight  = 20,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 0.1,
    random_state      = 42,
    n_jobs            = -1,
    eval_metric       = 'auc',
    verbosity         = 0,
    # device            = 'cuda'
)

lgb_model.fit(X_train.values, y_train.values)
xgb_model.fit(X_train.values, y_train.values)

# --- LightGBM Evaluation ---
y_prob_test_lgb = lgb_model.predict_proba(X_test.values)[:, 1]
y_prob_train_lgb = lgb_model.predict_proba(X_train.values)[:, 1]

print("ROC AUC Score - LightGBM")
print(f"Testing:  {roc_auc_score(y_test, y_prob_test_lgb):.4f}")
print(f"Training: {roc_auc_score(y_train, y_prob_train_lgb):.4f}")

print("-" * 30)

# --- XGBoost Evaluation ---
y_prob_test_xgb = xgb_model.predict_proba(X_test.values)[:, 1]
y_prob_train_xgb = xgb_model.predict_proba(X_train.values)[:, 1]

print("ROC AUC Score - XGBoost")
print(f"Testing:  {roc_auc_score(y_test, y_prob_test_xgb):.4f}")
print(f"Training: {roc_auc_score(y_train, y_prob_train_xgb):.4f}")

[LightGBM] [Info] Number of positive: 61167, number of negative: 246231
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.027308 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2809
[LightGBM] [Info] Number of data points in the train set: 307398, number of used features: 25
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.198983 -> initscore=-1.392662
[LightGBM] [Info] Start training from score -1.392662


C:\Users\admin\Desktop\kaggle_competition\comp_env\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\admin\Desktop\kaggle_competition\comp_env\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


ROC AUC Score - LightGBM
Testing:  0.9500
Training: 0.9965
------------------------------
ROC AUC Score - XGBoost
Testing:  0.9495
Training: 0.9633


In [93]:
cat_model = CatBoostClassifier(
     iterations        = 2000,
    learning_rate     = 0.02,
    depth             = 8,
    l2_leaf_reg       = 3,
    random_seed       = 42,
    eval_metric       = 'AUC',
    verbose           = 200
)

cat_model.fit(X_train, y_train)

y_prob_test_cat = cat_model.predict_proba(X_test)[:, 1]
y_prob_train_cat = cat_model.predict_proba(X_train)[:, 1]

print("ROC AUC Score - CatBoost")
print(f"Testing:  {roc_auc_score(y_test, y_prob_test_cat):.4f}")
print(f"Training: {roc_auc_score(y_train, y_prob_train_cat):.4f}")

0:	total: 69.3ms	remaining: 2m 18s
200:	total: 15.5s	remaining: 2m 18s
400:	total: 29.9s	remaining: 1m 59s
600:	total: 46.2s	remaining: 1m 47s
800:	total: 1m	remaining: 1m 31s
1000:	total: 1m 15s	remaining: 1m 15s
1200:	total: 1m 43s	remaining: 1m 8s
1400:	total: 1m 57s	remaining: 50.3s
1600:	total: 2m 11s	remaining: 32.8s
1800:	total: 2m 25s	remaining: 16.1s
1999:	total: 2m 42s	remaining: 0us
ROC AUC Score - CatBoost
Testing:  0.9480
Training: 0.9565


## Target Encoding - OOF

In [94]:
encoded_cat_cols = [col for col in train_df_scaled.columns if '_en' in col.lower() and '_engg' not in col.lower()]
encoded_cat_cols 

['Driver_en', 'Race_en']

In [96]:
from sklearn.model_selection import KFold

train_df_scaled_v1 = train_df_scaled.copy()
test_df_scaled_v1 = test_df_scaled.copy()

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for col in encoded_cat_cols:
    train_df_scaled_v1[f'{col}_mean'] = 0.0
    train_df_scaled_v1[f'{col}_std'] = 0.0

for train_idx, val_idx in kf.split(train_df_scaled_v1):
    train_fold = train_df_scaled_v1.iloc[train_idx]
    val_fold = train_df_scaled_v1.iloc[val_idx]

    for col in encoded_cat_cols:
        stats = train_fold.groupby(col)[target_col[0]].agg(['mean', 'std'])
        val_fold = val_fold.merge(stats, on=col, how='left')

        train_df_scaled_v1.loc[val_idx, f'{col}_mean'] = val_fold['mean'].values
        train_df_scaled_v1.loc[val_idx, f'{col}_std'] = val_fold['std'].values

        val_fold.drop(['mean', 'std'], axis=1, inplace=True)

for col in encoded_cat_cols:
    train_df_scaled_v1[[f'{col}_mean', f'{col}_std']] = train_df_scaled_v1[
        [f'{col}_mean', f'{col}_std']
    ].fillna(0)

for col in encoded_cat_cols:
    stats = train_df_scaled_v1.groupby(col)[target_col[0]].agg(['mean', 'std'])
    test_df_scaled_v1 = test_df_scaled_v1.merge(stats, on=col, how='left')

    test_df_scaled_v1[f'{col}_mean'] = test_df_scaled_v1['mean']
    test_df_scaled_v1[f'{col}_std'] = test_df_scaled_v1['std']

    test_df_scaled_v1.drop(['mean', 'std'], axis=1, inplace=True)

for col in encoded_cat_cols:
    test_df_scaled_v1[[f'{col}_mean', f'{col}_std']] = test_df_scaled_v1[
        [f'{col}_mean', f'{col}_std']
    ].fillna(0)


X, y = train_df_scaled_v1.drop(['id', 'PitNextLap'], axis='columns'), train_df_scaled_v1['PitNextLap'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

In [87]:
count_neg = (y == 0).sum()
count_pos = (y == 1).sum()
scale_weight = count_neg / count_pos

xgb_model = XGBClassifier(
    n_estimators      = 2000,
    learning_rate     = 0.02,
    max_depth         = 7,
    min_child_weight  = 20,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 0.1,
    random_state      = 42,
    n_jobs            = -1,
    eval_metric       = 'auc',
    verbosity         = 0
)

xgb_model.fit(X_train.values, y_train.values)

# --- XGBoost Evaluation ---
y_prob_test_xgb = xgb_model.predict_proba(X_test.values)[:, 1]
y_prob_train_xgb = xgb_model.predict_proba(X_train.values)[:, 1]

print("ROC AUC Score - XGBoost")
print(f"Testing:  {roc_auc_score(y_test, y_prob_test_xgb):.4f}")
print(f"Training: {roc_auc_score(y_train, y_prob_train_xgb):.4f}")

ROC AUC Score - XGBoost
Testing:  0.9498
Training: 0.9644


In [89]:
xgb_model = XGBClassifier(
    n_estimators      = 2000,
    learning_rate     = 0.02,
    max_depth         = 7,
    min_child_weight  = 20,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 0.1,
    random_state      = 42,
    n_jobs            = -1,
    eval_metric       = 'auc',
    verbosity         = 0,
)

xgb_model.fit(X.values, y.values)
submission(xgb_model, test_df_scaled_v1, file_name="First_XGB_solution.csv")

!!!!!!!!!!!!!!!!!! Removed id column !!!!!!!!!!!!!!!!!!!
Submissions saved to First_XGB_solution.csv path!!!!!!!!!!!!!!


In [97]:
lgb_model = LGBMClassifier(
    n_estimators      = 2000,
    learning_rate     = 0.02,
    num_leaves        = 255,
    min_child_samples = 20,
    subsample         = 0.8,
    subsample_freq    = 1,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 0.1,
    random_state      = 42,
    n_jobs            = -1
)


lgb_model.fit(X_train.values, y_train.values)

# --- LightGBM Evaluation ---
y_prob_test_lgb = lgb_model.predict_proba(X_test.values)[:, 1]
y_prob_train_lgb = lgb_model.predict_proba(X_train.values)[:, 1]

print("ROC AUC Score - LightGBM")
print(f"Testing:  {roc_auc_score(y_test, y_prob_test_lgb):.4f}")
print(f"Training: {roc_auc_score(y_train, y_prob_train_lgb):.4f}")

[LightGBM] [Info] Number of positive: 61167, number of negative: 246231
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.032061 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3581
[LightGBM] [Info] Number of data points in the train set: 307398, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.198983 -> initscore=-1.392662
[LightGBM] [Info] Start training from score -1.392662


C:\Users\admin\Desktop\kaggle_competition\comp_env\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\admin\Desktop\kaggle_competition\comp_env\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


ROC AUC Score - LightGBM
Testing:  0.9501
Training: 0.9974


In [98]:
lgb_model = LGBMClassifier(
    n_estimators      = 2000,
    learning_rate     = 0.02,
    num_leaves        = 255,
    min_child_samples = 20,
    subsample         = 0.8,
    subsample_freq    = 1,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 0.1,
    random_state      = 42,
    n_jobs            = -1
)
lgb_model.fit(X.values, y.values)
submission(lgb_model, test_df_scaled_v1, file_name="First_LGB_solution.csv")

[LightGBM] [Info] Number of positive: 87381, number of negative: 351759
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.023518 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3580
[LightGBM] [Info] Number of data points in the train set: 439140, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.198982 -> initscore=-1.392668
[LightGBM] [Info] Start training from score -1.392668
!!!!!!!!!!!!!!!!!! Removed id column !!!!!!!!!!!!!!!!!!!
Submissions saved to First_LGB_solution.csv path!!!!!!!!!!!!!!


In [102]:
cat_model_oot = CatBoostClassifier(
    # iterations=20000,
    eval_metric = "AUC",
    verbose=0,
    auto_class_weights='Balanced'
)

cat_model_oot.fit(X_train, y_train)

y_prob_test_cat = cat_model_oot.predict_proba(X_test)[:, 1]
y_prob_train_cat = cat_model_oot.predict_proba(X_train)[:, 1]

print("ROC AUC Score - CatBoost")
print(f"Testing:  {roc_auc_score(y_test, y_prob_test_cat):.4f}")
print(f"Training: {roc_auc_score(y_train, y_prob_train_cat):.4f}")

ROC AUC Score - CatBoost
Testing:  0.9484
Training: 0.9595


In [ ]:
# cat_model_oot = CatBoostClassifier(
#     iterations=40000,
#     # learning_rate=0.1,
#     # depth=6,
#     verbose=0,
#     # random_seed=42,
#     auto_class_weights='Balanced'
# )
#
# cat_model_oot.fit(X, y)
# submission(cat_model_oot, test_df_scaled_v1, file_name="FE_v16_catboost_target_encoding_40K_iterations.csv")

In [ ]:
# cat_model = CatBoostClassifier(
#     iterations        = 2000,
#     learning_rate     = 0.02,
#     depth             = 8,
#     l2_leaf_reg       = 3,
#     random_seed       = 42,
#     task_type         = 'GPU',
#     eval_metric       = 'AUC',
#     verbose           = 200
# )